In [ ]:
library(Seurat)
library(dplyr)
library(tidyr)
library(tibble)
library(ggplot2)
library(rstatix)
library(ggrepel)
library(grid)

ggtheme <- function() {
  theme(
    axis.text = element_text(size = 20),
    axis.title = element_text(size = 20),
    text = element_text(size = 20, colour = "black"),
    legend.text = element_text(size = 20),
    legend.key.size =  unit(10, units = "points")
    
  )
}


# Compositional analysis

In [ ]:
# ============================================================
# Prepare sample-level cell population proportions
# ============================================================

# Extract Seurat metadata
md <- gut_cdn@meta.data


# Calculate cluster frequencies per sample
df4 <- md %>%
  group_by(data_folder, integrated_snn_res.0.6) %>%
  summarise(n = n()) %>%
  unique() %>%
  mutate(freq = n / sum(n)) %>%
  dplyr::select(-n) %>%
  pivot_wider(
    names_from = integrated_snn_res.0.6,
    values_from = freq,
    values_fill = 0
  ) %>%
  pivot_longer(
    cols = !data_folder,
    names_to = "integrated_snn_res.0.6",
    values_to = "freq"
  )


# Add sample metadata back after reshaping
md_to_join <- md %>%
  dplyr::select(
    diagnosis,
    antitnf_response,
    timepoint,
    tissue,
    layer,
    data_folder,
    age,
    sex
  ) %>%
  unique()

df4 <- left_join(
  df4,
  md_to_join,
  by = "data_folder"
)


# ============================================================
# CLR transformation of compositional data
# ============================================================

# Calculate cluster proportions per sample
df_proportions <- md %>%
  group_by(data_folder, integrated_snn_res.0.6) %>%
  summarise(n = n()) %>%
  unique() %>%
  mutate(freq = n / sum(n)) %>%
  dplyr::select(-n) %>%
  pivot_wider(
    names_from = integrated_snn_res.0.6,
    values_from = freq,
    values_fill = 0.0001
  ) %>%
  column_to_rownames("data_folder")


# Geometric mean function used for CLR transformation
geom_mean <- function(x) {
  exp(mean(log(x)))
}


# Centered log-ratio transformation
clr_transform <- function(x) {
  log(x / geom_mean(x))
}


# Apply CLR transformation sample-wise
clr_df <- t(apply(df_proportions, 1, clr_transform))

clr_df <- as.data.frame(clr_df)

colnames(clr_df) <- paste0(
  "cluster",
  colnames(clr_df)
)


# Add metadata back
clr_df2 <- clr_df %>%
  rownames_to_column("data_folder") %>%
  left_join(
    md_to_join,
    by = "data_folder"
  )


# ============================================================
# Prepare data for statistical testing
# ============================================================

df_filt <- clr_df2 %>%
  dplyr::filter(
    diagnosis %in% c("CD", "HEA", "UC", "FGID")
  ) %>%
  pivot_longer(
    cols = starts_with("cluster"),
    names_to = "seurat_clusters",
    values_to = "freq_clr"
  ) %>%
  mutate(
    seurat_clusters = gsub(
      "cluster",
      "",
      seurat_clusters
    )
  )


# ============================================================
# Wilcoxon tests comparing each diagnosis against all others
# ============================================================

diag <- c("CD", "UC", "FGID", "HEA")
tissue <- c("ILE", "COL", "DUO")
layer <- c("LP", "EPI")


# Loop through all diagnosis/tissue/layer combinations
for (i in diag) {
  for (j in tissue) {
    for (k in layer) {

      stat_test <- df_filt %>%
        mutate(
          diagnosis = if_else(
            diagnosis == i,
            i,
            "Other"
          )
        ) %>%
        dplyr::filter(
          tissue == j &
          layer == k
        ) %>%
        group_by(seurat_clusters) %>%
        rstatix::wilcox_test(
          freq_clr ~ diagnosis,
          detailed = TRUE,
          ref.group = "Other"
        )

      # Store comparison metadata
      stat_test$tissue <- j
      stat_test$diag <- i
      stat_test$layer <- k


      # Combine all results
      if (
        i == diag[1] &
        j == tissue[1] &
        k == layer[1]
      ) {
        stat_all <- stat_test
      } else {
        stat_all <- rbind(
          stat_all,
          stat_test
        )
      }
    }
  }
}


# ============================================================
# Multiple testing correction
# ============================================================

# Bonferroni correction for:
# 11 clusters x 3 tissues x 2 layers
stat_all <- stat_all %>%
  mutate(
    p.adj = if_else(
      p * 11 * 3 * 2 < 0.05,
      p * 11 * 3 * 2,
      1
    )
  )


# ============================================================
# Visualization
# ============================================================

options(
  repr.plot.width = 12,
  repr.plot.height = 7
)


stat_all %>%
  arrange(p) %>%
  mutate(
    estimate = -estimate,
    delabel = if_else(
      p.adj < 0.05,
      paste0(
        seurat_clusters,
        " in ",
        group2
      ),
      ""
    )
  ) %>%
  ggplot(
    aes(
      x = estimate,
      y = -log(p),
      label = delabel
    )
  ) +
  theme_bw() +
  geom_point(
    aes(
      color = group2,
      shape = seurat_clusters
    ),
    size = 3
  ) +
  ggrepel::geom_label_repel() +
  facet_grid(
    cols = vars(tissue),
    rows = vars(layer)
  ) +
  scale_fill_manual() +
  scale_shape_manual(
    values = c(0:10)
  ) +
  geom_hline(
    yintercept = -log(0.05 / (11 * 3 * 2))
  ) +
  geom_vline(
    xintercept = 0
  ) +
  ggtheme()


# Save figure
ggsave(
  "figures/boxplot_freq_gut_5p_24/volcano_all_freq.svg",
  width = 13,
  height = 7
)